In [25]:
import pandas as pd #import library pandas มาใช้งานสำหรับจัดการ dataframe ร่วมกับ numpy
import numpy as np #import library numpy มาใช้งานสำหรับคำนวณ
from sklearn.preprocessing import LabelEncoder, StandardScaler

df = pd.read_csv('StudentsPerformance.csv') #อ่านไฟล์

#ดู 5 แถวแรกของข้อมูล
df.head()

,gender,race/ethnicity,parental level of education,lunch,test preparation course,math score,reading score,writing score
0,female,group B,bachelor's degree,standard,none,72,72,74
1,female,group C,some college,standard,completed,69,90,88
2,female,group B,master's degree,standard,none,90,95,93
3,male,group A,associate's degree,free/reduced,none,47,57,44
4,male,group C,some college,standard,none,76,78,75


In [26]:
#ดูว่ามี column ไหนว่างบ้าง เพื่อที่จะทำ Data cleansing
df.isnull().any()

gender                         False
race/ethnicity                 False
parental level of education    False
lunch                          False
test preparation course        False
math score                     False
reading score                  False
writing score                  False
dtype: bool

In [27]:
#ลบข้อมูลที่ซ้ำกันทุกแถว
df = df.drop_duplicates()
#reset ค่า index ใหม่ กัน index กระโดด
df = df.reset_index(drop=True)

#ดูว่ายังมีข้อมูลซ้ำกันอีกไหม
df.duplicated().sum()

np.int64(0)

In [28]:
# ทำ feature engineering

# เพิ่ม feature ใหม่ เพื่่อให้ model เรียนรู้ได้ดีขึ้น
df["average_score"] = (df["math score"] + df["reading score"] + df["writing score"]) / 3
df["language_score"] = (df["reading score"] + df["writing score"]) / 2
df["stem_strength"] = (df["math score"] - df["language_score"])
df["reading_writing_diff"] = abs(df["reading score"] - df["writing score"])

#เพิ่ม feature ใหม่ สำหรับการผ่านแต่ละอย่าง
df["pass_math"] = (df["math score"] >= 50).astype(int)
df["pass_reading"] = (df["reading score"] >= 50).astype(int)
df["pass_writing"] = (df["writing score"] >= 50).astype(int)
# หากคะแนนผ่านทุกวิชา
df["pass_all"] = ((df["math score"] >= 50) & (df["reading score"] >= 50) & (df["writing score"] >= 50)).astype(int)

# เพิ่ม feature target (สิ่งที่ต้องการทำนาย)
df["final_pass"] = (df["average_score"] >= 50).astype(int)

df["gender"] = df["gender"].map({"male":0, "female":1})
df["lunch_quality"] = df["lunch"].map({"standard":1, "free/reduced":0})
df["prepared"] = df["test preparation course"].map({"none":0, "completed":1})
df["parental level of education"] = df["parental level of education"].map({"some high school" : 0, "high school" : 1, "some college" : 2, "associate's degree" : 3, "bachelor's degree" : 4, "master's degree" : 5})
df["race/ethnicity"] = df["race/ethnicity"].map({"group A" : 0, "group B" : 1, "group C" : 2, "group D" : 3, "group E" : 4})

df.drop(columns=["lunch","test preparation course"], inplace=True)

df.head()

,gender,race/ethnicity,parental level of education,math score,reading score,writing score,average_score,language_score,stem_strength,reading_writing_diff,pass_math,pass_reading,pass_writing,pass_all,final_pass,lunch_quality,prepared
0,1,1,4,72,72,74,72.666667,73.0,-1.0,2,1,1,1,1,1,1,0
1,1,2,2,69,90,88,82.333333,89.0,-20.0,2,1,1,1,1,1,1,1
2,1,1,5,90,95,93,92.666667,94.0,-4.0,2,1,1,1,1,1,1,0
3,0,0,3,47,57,44,49.333333,50.5,-3.5,13,0,1,0,0,0,0,0
4,0,2,2,76,78,75,76.333333,76.5,-0.5,3,1,1,1,1,1,1,0


In [29]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

#ป้องกัน data leakage
X = df.drop(columns=["final_pass", "average_score", "pass_all", "math score", "reading score", "writing score", "pass_math", "pass_reading", "pass_writing"])
y = df["final_pass"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

#สร้าง model ANN

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

model = Sequential([
    Dense(32, activation="relu", input_shape=(X_train.shape[1],)),
    Dense(16, activation="relu"),
    Dense(8, activation="relu"),
    Dense(1, activation="sigmoid")
])

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.fit(
    X_train,
    y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.2
)

loss, accuracy = model.evaluate(X_test, y_test)

print("Accuracy:", accuracy)

Epoch 1/50


c:\Users\Neide\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.8578 - loss: 0.5738 - val_accuracy: 0.8875 - val_loss: 0.5058
Epoch 2/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9094 - loss: 0.4391 - val_accuracy: 0.8938 - val_loss: 0.3961
Epoch 3/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9078 - loss: 0.3383 - val_accuracy: 0.9000 - val_loss: 0.3193
Epoch 4/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9062 - loss: 0.2756 - val_accuracy: 0.9000 - val_loss: 0.2753
Epoch 5/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9078 - loss: 0.2386 - val_accuracy: 0.9000 - val_loss: 0.2472
Epoch 6/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9078 - loss: 0.2142 - val_accuracy: 0.9000 - val_loss: 0.2241
Epoch 7/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9109 - loss: 0.1968 - val_accuracy: 0.9000 - val_loss: 0.2045
Epoch 8/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9109 - loss: 0.1822 - val_accuracy: 0.9000 - val_loss: 0.1906
Epo

In [30]:
import pickle

# บันทึก model
model.save("ann_model.keras")

# บันทึก scaler
with open("ann_scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

print("✅ บันทึก ann_model.keras และ ann_scaler.pkl เรียบร้อย")

✅ บันทึก ann_model.keras และ ann_scaler.pkl เรียบร้อย
